<a href="https://colab.research.google.com/github/snur25/akbank-ml-housing-prediction/blob/main/butce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import React, { useState, useEffect } from 'react';
import { View, Text, TouchableOpacity, StyleSheet, ScrollView, SafeAreaView, Dimensions, Alert, TextInput, ActivityIndicator, Platform, StatusBar } from 'react-native';
import AsyncStorage from '@react-native-async-storage/async-storage';

// --- SABİTLER ---
const SCREEN_WIDTH = Dimensions.get('window').width;
const DAY_LABELS = ['Pzt', 'Sal', 'Çar', 'Per', 'Cum', 'Cmt', 'Paz'];
const STORAGE_KEY = '@finans_simple_v7';

// Pastel Renkler
const COLORS = {
  green: '#81C784', // Pastel Yeşil
  red: '#E57373',   // Pastel Kırmızı
  bg: '#FAFAFA',
  text: '#212121',
  subText: '#9E9E9E',
  cardBg: '#FFFFFF'
};

export default function App() {
  const [loading, setLoading] = useState(true);
  const [activeTab, setActiveTab] = useState('calculator');

  // TARİH VE VERİLER
  const [viewDate, setViewDate] = useState(new Date());
  const [noSpendDays, setNoSpendDays] = useState({});
  const [transactions, setTransactions] = useState([]);

  // Sabitler
  const [fixedExpenses, setFixedExpenses] = useState([]);
  const [fixedIncomes, setFixedIncomes] = useState([]);

  // FORM GİRDİLERİ
  const [calcInput, setCalcInput] = useState('0');

  // Gider Formu
  const [newFixedTitle, setNewFixedTitle] = useState('');
  const [newFixedAmount, setNewFixedAmount] = useState('');
  const [showFixedForm, setShowFixedForm] = useState(false);

  // Gelir Formu
  const [newIncomeTitle, setNewIncomeTitle] = useState('');
  const [newIncomeAmount, setNewIncomeAmount] = useState('');
  const [newIncomeDay, setNewIncomeDay] = useState('');
  const [showIncomeForm, setShowIncomeForm] = useState(false);

  // --- HESAPLAMA MOTORU (STANDART TAKVİM AYI) ---
  const today = new Date();
  today.setHours(0,0,0,0);

  // Dönem: Ayın 1'i ile Ayın Son Günü
  const getCurrentPeriod = () => {
    const now = new Date();
    const start = new Date(now.getFullYear(), now.getMonth(), 1);
    const end = new Date(now.getFullYear(), now.getMonth() + 1, 0); // Ayın son günü

    start.setHours(0,0,0,0);
    end.setHours(23,59,59,999);
    return { start, end };
  };

  const { start: periodStart, end: periodEnd } = getCurrentPeriod();

  const getRemainingDays = () => {
    const diffTime = periodEnd.getTime() - today.getTime();
    const diffDays = Math.ceil(diffTime / (1000 * 60 * 60 * 24));
    return diffDays >= 0 ? diffDays + 1 : 0; // Bugün dahil +1
  };

  const remainingDaysTotal = getRemainingDays();

  const getFutureNoSpendCount = () => {
    let count = 0;
    const tempDate = new Date(today);
    while (tempDate <= periodEnd) {
      const key = `${tempDate.getFullYear()}-${tempDate.getMonth()}-${tempDate.getDate()}`;
      if (noSpendDays[key]) count++;
      tempDate.setDate(tempDate.getDate() + 1);
    }
    return count;
  };

  const effectiveRemainingDays = Math.max(1, remainingDaysTotal - getFutureNoSpendCount());

  // Bakiye Hesaplama
  const calculateBalance = () => {
    // 1. Nakit İşlemler (Bu ay yapılanlar)
    const periodTrans = transactions.filter(t => {
      const tDate = new Date(t.date);
      return tDate >= periodStart && tDate <= periodEnd;
    });

    const cashIncome = periodTrans.filter(t => t.type === 'income').reduce((acc, t) => acc + t.amount, 0);
    const cashExpense = periodTrans.filter(t => t.type === 'expense').reduce((acc, t) => acc + t.amount, 0);

    // 2. Sabitler (Her ay otomatik eklenir/düşülür)
    const totalFixedExpense = fixedExpenses.reduce((acc, t) => acc + t.amount, 0);
    const totalFixedIncome = fixedIncomes.reduce((acc, t) => acc + t.amount, 0);

    return (cashIncome + totalFixedIncome) - (cashExpense + totalFixedExpense);
  };

  const netBalance = calculateBalance();
  const dailySafeLimit = netBalance / effectiveRemainingDays;

  // --- STORAGE ---
  useEffect(() => { loadData(); }, []);
  useEffect(() => { if (!loading) saveData(); }, [transactions, fixedExpenses, fixedIncomes, noSpendDays]);

  const saveData = async () => {
    try {
      const data = { transactions, fixedExpenses, fixedIncomes, noSpendDays };
      await AsyncStorage.setItem(STORAGE_KEY, JSON.stringify(data));
    } catch (e) { console.log(e); }
  };

  const loadData = async () => {
    try {
      const json = await AsyncStorage.getItem(STORAGE_KEY);
      if (json) {
        const data = JSON.parse(json);
        setTransactions((data.transactions || []).map(t => ({...t, date: new Date(t.date)})));
        setFixedExpenses(data.fixedExpenses || []);
        setFixedIncomes(data.fixedIncomes || []);
        setNoSpendDays(data.noSpendDays || {});
      }
    } catch (e) { console.log(e); }
    finally { setLoading(false); }
  };

  // --- İŞLEMLER ---
  const handleAddTransaction = (type) => {
    const amount = parseFloat(calcInput);
    if (amount > 0) {
      setTransactions(prev => [...prev, { id: Date.now(), type, amount, date: new Date() }]);
      setCalcInput('0');
      Alert.alert("Başarılı", `${type === 'income' ? 'Gelir' : 'Harcama'} eklendi.`);
    }
  };

  const handleCalcPress = (k) => {
    if(k==='C') setCalcInput('0');
    else if(k==='.') { if(!calcInput.includes('.')) setCalcInput(p=>p+'.'); }
    else setCalcInput(p => p==='0'?String(k):p+k);
  };

  const handleAddFixedExpense = () => {
    if(newFixedTitle && newFixedAmount) {
      setFixedExpenses(p => [...p, {id: Date.now(), title: newFixedTitle, amount: parseFloat(newFixedAmount)}]);
      setNewFixedTitle(''); setNewFixedAmount(''); setShowFixedForm(false);
    }
  };

  const handleAddFixedIncome = () => {
    if(newIncomeTitle && newIncomeAmount) {
      setFixedIncomes(p => [...p, {
        id: Date.now(),
        title: newIncomeTitle,
        amount: parseFloat(newIncomeAmount),
        day: newIncomeDay || '1'
      }]);
      setNewIncomeTitle(''); setNewIncomeAmount(''); setNewIncomeDay(''); setShowIncomeForm(false);
    }
  };

  // --- TAKVİM ---
  const generateCalendarDays = () => {
    const year = viewDate.getFullYear();
    const month = viewDate.getMonth();
    const daysInMonth = new Date(year, month + 1, 0).getDate();
    const firstDay = new Date(year, month, 1).getDay();
    const startingEmpty = firstDay === 0 ? 6 : firstDay - 1;

    const days = [];
    for (let i = 0; i < startingEmpty; i++) days.push({ type: 'empty', id: `e-${i}` });
    for (let i = 1; i <= daysInMonth; i++) days.push({ type: 'day', day: i, id: `d-${i}` });
    return days;
  };

  if (loading) return <ActivityIndicator style={{flex:1}} />;

  // --- EKRAN 1: HESAP MAKİNESİ ---
  const renderCalculator = () => (
    <View style={styles.calcContainer}>
      <View style={styles.calcHeaderInfo}>
         <Text style={styles.calcInfoTitle}>Bugün Harcayabilirsin</Text>
         <Text style={[styles.calcInfoVal, dailySafeLimit<0 && {color: COLORS.red}]}>
            {dailySafeLimit.toFixed(2)} ₺
         </Text>
      </View>

      <View style={styles.disp}><Text style={styles.dispText}>{calcInput} ₺</Text></View>

      <View style={styles.pad}>
        {[[1,2,3],[4,5,6],[7,8,9]].map((r,i)=>(
          <View key={i} style={styles.padRow}>{r.map(n=><TouchableOpacity key={n} style={styles.btn} onPress={()=>handleCalcPress(n)}><Text style={styles.btnTxt}>{n}</Text></TouchableOpacity>)}</View>
        ))}
        <View style={styles.padRow}>
          <TouchableOpacity style={[styles.btn,{backgroundColor:'#FFEBEE'}]} onPress={()=>handleCalcPress('C')}><Text style={[styles.btnTxt,{color: COLORS.red}]}>C</Text></TouchableOpacity>
          <TouchableOpacity style={styles.btn} onPress={()=>handleCalcPress(0)}><Text style={styles.btnTxt}>0</Text></TouchableOpacity>
          <TouchableOpacity style={styles.btn} onPress={()=>handleCalcPress('.')}><Text style={styles.btnTxt}>.</Text></TouchableOpacity>
        </View>
      </View>

      <View style={styles.actRow}>
        <TouchableOpacity style={[styles.actBtn,{backgroundColor: COLORS.green}]} onPress={()=>handleAddTransaction('income')}>
          <Text style={styles.actTxt}>+ Gelir</Text>
        </TouchableOpacity>
        <TouchableOpacity style={[styles.actBtn,{backgroundColor: COLORS.red}]} onPress={()=>handleAddTransaction('expense')}>
          <Text style={styles.actTxt}>- Harca</Text>
        </TouchableOpacity>
      </View>
    </View>
  );

  // --- EKRAN 2: DASHBOARD ---
  const renderDashboard = () => (
    <ScrollView style={styles.scroll} showsVerticalScrollIndicator={false}>
      {/* Header (Sadece Tarih, Ayar ikonu yok) */}
      <View style={styles.topBar}>
        <Text style={styles.welcome}>Aylık Bütçe</Text>
        <Text style={styles.periodText}>
          {periodStart.toLocaleDateString('tr-TR', {day:'numeric', month:'long'})} - {periodEnd.toLocaleDateString('tr-TR', {day:'numeric', month:'long'})}
        </Text>
      </View>

      {/* Limit Kartı */}
      <View style={styles.limitCard}>
        <Text style={styles.cardLabel}>Günlük Harcanabilir Limit</Text>
        <Text style={[styles.cardValue, dailySafeLimit < 0 && {color: COLORS.red}]}>
          {dailySafeLimit.toFixed(2)} ₺
        </Text>
        <View style={styles.cardFooter}>
          <Text style={styles.cardSub}>Net Bakiye: {netBalance.toFixed(0)} ₺</Text>
          <Text style={styles.cardSub}>Kalan Gün: {effectiveRemainingDays}</Text>
        </View>
        <View style={styles.progressBarBg}>
           <View style={[styles.progressBarFill, {width: `${Math.min(100, (30-remainingDaysTotal)/30*100)}%`}]} />
        </View>
        <Text style={styles.progressText}>Ayın bitmesine {remainingDaysTotal} gün kaldı</Text>
      </View>

      {/* Sabit Gelirler */}
      <View style={styles.section}>
        <View style={styles.secHeader}>
          <Text style={styles.secTitle}>Sabit Gelirler</Text>
          <TouchableOpacity onPress={() => setShowIncomeForm(!showIncomeForm)}>
            <Text style={styles.linkText}>{showIncomeForm?'Kapat':'+ Ekle'}</Text>
          </TouchableOpacity>
        </View>

        {showIncomeForm && (
          <View style={styles.form}>
            <TextInput placeholder="Gelir Adı" style={styles.input} value={newIncomeTitle} onChangeText={setNewIncomeTitle}/>
            <View style={{flexDirection:'row', gap:10}}>
              <TextInput placeholder="Tutar" keyboardType="numeric" style={[styles.input, {flex:1}]} value={newIncomeAmount} onChangeText={setNewIncomeAmount}/>
              <TextInput placeholder="Gün (1-31)" keyboardType="numeric" style={[styles.input, {width:100}]} value={newIncomeDay} onChangeText={setNewIncomeDay}/>
            </View>
            <TouchableOpacity style={styles.saveBtn} onPress={handleAddFixedIncome}><Text style={styles.saveBtnText}>Kaydet</Text></TouchableOpacity>
          </View>
        )}

        {fixedIncomes.map(f => (
          <View key={f.id} style={styles.row}>
            <View>
              <Text style={styles.rowTitle}>{f.title}</Text>
              <Text style={styles.rowDateLabel}>Ayın {f.day}. günü</Text>
            </View>
            <View style={{flexDirection:'row'}}>
              <Text style={[styles.rowAmount, {color: COLORS.green}]}>+{f.amount} ₺</Text>
              <TouchableOpacity onPress={()=>setFixedIncomes(fixedIncomes.filter(x=>x.id!==f.id))}><Text style={{marginLeft:10}}>🗑️</Text></TouchableOpacity>
            </View>
          </View>
        ))}
      </View>

      {/* Sabit Giderler */}
      <View style={styles.section}>
        <View style={styles.secHeader}>
          <Text style={styles.secTitle}>Sabit Giderler</Text>
          <TouchableOpacity onPress={() => setShowFixedForm(!showFixedForm)}>
            <Text style={styles.linkText}>{showFixedForm?'Kapat':'+ Ekle'}</Text>
          </TouchableOpacity>
        </View>

        {showFixedForm && (
          <View style={styles.form}>
            <TextInput placeholder="Gider Adı" style={styles.input} value={newFixedTitle} onChangeText={setNewFixedTitle}/>
            <TextInput placeholder="Tutar" keyboardType="numeric" style={styles.input} value={newFixedAmount} onChangeText={setNewFixedAmount}/>
            <TouchableOpacity style={styles.saveBtn} onPress={handleAddFixedExpense}><Text style={styles.saveBtnText}>Kaydet</Text></TouchableOpacity>
          </View>
        )}

        {fixedExpenses.map(f => (
          <View key={f.id} style={styles.row}>
            <Text style={styles.rowTitle}>{f.title}</Text>
            <View style={{flexDirection:'row'}}>
              <Text style={[styles.rowAmount, {color: COLORS.red}]}>-{f.amount} ₺</Text>
              <TouchableOpacity onPress={()=>setFixedExpenses(fixedExpenses.filter(x=>x.id!==f.id))}><Text style={{marginLeft:10}}>🗑️</Text></TouchableOpacity>
            </View>
          </View>
        ))}
      </View>

      {/* Takvim */}
      <View style={styles.section}>
        <View style={styles.calHeader}>
          <TouchableOpacity onPress={()=>setViewDate(new Date(viewDate.getFullYear(), viewDate.getMonth()-1, 1))}><Text style={styles.arrow}>{'<'}</Text></TouchableOpacity>
          <Text style={styles.calTitle}>{viewDate.toLocaleDateString('tr-TR', {month:'long', year:'numeric'})}</Text>
          <TouchableOpacity onPress={()=>setViewDate(new Date(viewDate.getFullYear(), viewDate.getMonth()+1, 1))}><Text style={styles.arrow}>{'>'}</Text></TouchableOpacity>
        </View>

        <View style={styles.daysRow}>{DAY_LABELS.map(d=><Text key={d} style={styles.dayLabel}>{d}</Text>)}</View>

        <View style={styles.grid}>
          {generateCalendarDays().map((item, i) => {
             if (item.type==='empty') return <View key={item.id} style={styles.boxEmpty} />;
             const thisDay = new Date(viewDate.getFullYear(), viewDate.getMonth(), item.day);
             const isToday = thisDay.getDate() === today.getDate() && thisDay.getMonth() === today.getMonth() && thisDay.getFullYear() === today.getFullYear();
             const noSpendKey = `${thisDay.getFullYear()}-${thisDay.getMonth()}-${thisDay.getDate()}`;
             const isNoSpend = noSpendDays[noSpendKey];

             return (
               <TouchableOpacity
                 key={item.id}
                 style={[styles.box, isToday && styles.boxToday, isNoSpend && styles.boxActive]}
                 onPress={() => setNoSpendDays(p => ({...p, [noSpendKey]: !p[noSpendKey]}))}
               >
                 <Text style={[styles.boxText, isToday&&styles.textToday, isNoSpend&&styles.textActive]}>{item.day}</Text>
               </TouchableOpacity>
             );
          })}
        </View>
      </View>

      {/* Son İşlemler */}
      <View style={[styles.section, {marginBottom:100}]}>
        <Text style={styles.secTitle}>Son İşlemler (Nakit)</Text>
        {transactions.slice().reverse().slice(0, 5).map(t => (
          <View key={t.id} style={styles.row}>
            <View>
              <Text style={styles.rowTitle}>{t.type==='income'?'Gelir':'Harcama'}</Text>
              <Text style={styles.rowDate}>{t.date.toLocaleDateString()} {t.date.getHours()}:{String(t.date.getMinutes()).padStart(2,'0')}</Text>
            </View>
            <View style={{flexDirection:'row', alignItems:'center'}}>
              <Text style={{fontWeight:'bold', color:t.type==='income'? COLORS.green : COLORS.red}}>{t.type==='income'?'+':'-'}{t.amount}₺</Text>
              <TouchableOpacity onPress={()=>setTransactions(prev=>prev.filter(x=>x.id!==t.id))}><Text style={{marginLeft:10, color:'#CCC'}}>✕</Text></TouchableOpacity>
            </View>
          </View>
        ))}
      </View>
    </ScrollView>
  );

  return (
    <SafeAreaView style={styles.cont}>
      <StatusBar barStyle="dark-content" />
      <View style={{flex:1}}>{activeTab==='calculator'?renderCalculator():renderDashboard()}</View>

      {/* Nav Bar */}
      <View style={styles.nav}>
        <TouchableOpacity style={styles.navItem} onPress={()=>setActiveTab('calculator')}>
          <Text style={{fontSize:24, opacity:activeTab==='calculator'?1:0.3}}>🔢</Text>
          <Text style={{fontSize:10, color:activeTab==='calculator'?'black':'gray'}}>Hesapla</Text>
        </TouchableOpacity>
        <TouchableOpacity style={styles.navItem} onPress={()=>setActiveTab('dashboard')}>
          <Text style={{fontSize:24, opacity:activeTab==='dashboard'?1:0.3}}>📅</Text>
          <Text style={{fontSize:10, color:activeTab==='dashboard'?'black':'gray'}}>Takvim & Özet</Text>
        </TouchableOpacity>
      </View>
    </SafeAreaView>
  );
}

// --- STİLLER ---
const styles = StyleSheet.create({
  cont: {
    flex:1,
    backgroundColor: COLORS.bg,
    paddingTop: Platform.OS === 'android' ? 40 : 0
  },
  scroll: {paddingBottom:50},

  // Calculator
  calcContainer: {flex:1, justifyContent:'flex-end', paddingBottom:20},
  calcHeaderInfo: {position:'absolute', top:20, alignSelf:'center', alignItems:'center', zIndex:10},
  calcInfoTitle: {color: COLORS.subText, fontSize:12, fontWeight:'600'},
  calcInfoVal: {fontSize:24, fontWeight:'700', color: COLORS.text},
  disp: {flex:1, justifyContent:'center', alignItems:'flex-end', padding:30},
  dispText: {fontSize:50},
  pad: {paddingHorizontal:20},
  padRow: {flexDirection:'row', justifyContent:'space-between', marginBottom:12},
  btn: {width:80, height:80, borderRadius:40, backgroundColor: COLORS.cardBg, justifyContent:'center', alignItems:'center', elevation:2},
  btnTxt: {fontSize:26, fontWeight:'500'},
  actRow: {flexDirection:'row', paddingHorizontal:20, gap:15, marginTop:20},
  actBtn: {flex:1, height:60, borderRadius:30, justifyContent:'center', alignItems:'center'},
  actTxt: {color:'#fff', fontSize:18, fontWeight:'600'},

  // Dashboard Header
  topBar: {flexDirection:'column', justifyContent:'center', padding:20, paddingTop:10, alignItems:'flex-start'},
  welcome: {color: COLORS.subText, fontSize:12},
  periodText: {fontSize:20, fontWeight:'700', color: COLORS.text, marginTop:2},

  // Limit Card
  limitCard: {margin:20, marginTop:5, padding:25, backgroundColor: COLORS.cardBg, borderRadius:24, alignItems:'center', elevation:4, shadowColor:'#000', shadowOpacity:0.05, shadowRadius:10},
  cardLabel: {fontSize:12, color: COLORS.subText, fontWeight:'600', marginBottom:8},
  cardValue: {fontSize:38, fontWeight:'700', color: COLORS.text},
  cardFooter: {flexDirection:'row', width:'100%', justifyContent:'space-between', marginTop:15},
  cardSub: {fontSize:11, color: '#555', fontWeight:'500'},
  progressBarBg: {width:'100%', height:6, backgroundColor:'#F0F0F0', borderRadius:3, marginTop:15},
  progressBarFill: {height:'100%', backgroundColor: COLORS.green, borderRadius:3},
  progressText: {fontSize:10, color:'#AAA', marginTop:6, alignSelf:'flex-start'},

  // Sections
  section: {marginHorizontal:20, marginBottom:20},
  secHeader: {flexDirection:'row', justifyContent:'space-between', marginBottom:10},
  secTitle: {fontSize:18, fontWeight:'700'},
  linkText: {color:'#2196F3', fontWeight:'600'},
  form: {backgroundColor: COLORS.cardBg, padding:15, borderRadius:12, marginBottom:10},
  input: {backgroundColor:'#F5F5F5', padding:10, borderRadius:8, marginBottom:8},
  saveBtn: {backgroundColor:'#212121', padding:10, borderRadius:8, alignItems:'center'},
  saveBtnText: {color:'#fff', fontWeight:'600'},

  // Rows
  row: {flexDirection:'row', justifyContent:'space-between', alignItems:'center', padding:14, backgroundColor: COLORS.cardBg, borderRadius:12, marginBottom:6, borderBottomWidth:1, borderColor:'#f9f9f9'},
  rowTitle: {fontWeight:'600', color: COLORS.text},
  rowDateLabel: {fontSize:10, color: COLORS.subText},
  rowAmount: {fontWeight:'600'},
  rowDate: {fontSize:10, color:'#aaa'},

  // Calendar
  calHeader: {flexDirection:'row', justifyContent:'space-between', alignItems:'center', backgroundColor: COLORS.cardBg, padding:10, borderRadius:12, marginBottom:10},
  arrow: {fontSize:20, paddingHorizontal:10, color:'#555'},
  calTitle: {fontWeight:'700', fontSize:16},
  daysRow: {flexDirection:'row', marginBottom:5},
  dayLabel: {width:(SCREEN_WIDTH-40)/7, textAlign:'center', fontSize:11, color:'#aaa'},
  grid: {flexDirection:'row', flexWrap:'wrap'},
  box: {width:(SCREEN_WIDTH-46)/7, height:40, justifyContent:'center', alignItems:'center', margin:0.4, borderRadius:8},
  boxEmpty: {width:(SCREEN_WIDTH-46)/7, height:40, margin:0.4},
  boxToday: {borderWidth:2, borderColor:'#2196F3'},
  boxActive: {backgroundColor:'#212121'},
  boxText: {color:'#333'},
  textToday: {color:'#2196F3', fontWeight:'bold'},
  textActive: {color:'#fff'},

  // Navbar
  nav: {height:70, flexDirection:'row', backgroundColor:'#fff', borderTopWidth:1, borderColor:'#eee', justifyContent:'space-around', alignItems:'center'},
  navItem: {alignItems:'center'},
});

SyntaxError: unterminated string literal (detected at line 51) (ipython-input-3472436740.py, line 51)